# HEP classifier showcase: profiled $D_s$ with a tau-energy-scale nuisance

This notebook accompanies the doc page
[`usecases/hep`](../../docs/usecases/hep/index.md). It walks Door 3 (classifier to density
ratios to scores) on the FAIR Universe HiggsML public dataset (DOI
`10.5281/zenodo.15131565`, CC-BY-4.0): a signal-vs-background classifier gives the two rate
score columns, and a tau-energy-scale (`tes`) minus/plus classifier gives the one nuisance-shape
column that has no closed form.

Set `SCOREQUANT_EXAMPLE_FAST=1` to shrink every classifier and solver budget for a quick pass;
the committed fixture itself never shrinks.

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)

## The fixture and its provenance

1,000 row-aligned events at seven committed `tes` values, `tes` in `{0.90, 0.95, 0.975, 1.00,
1.025, 1.05, 1.10}`. The upstream selection is held fixed (`dopostprocess=False`), so every
variant keeps the same 1,000 rows -- the example measures the *shape* sensitivity to the tau
energy scale, not the acceptance sensitivity. `ztautau`, `ttbar`, and `diboson` are collapsed
into one background component: the committed sample carries only 26 `ttbar` and 4 `diboson`
rows, too few to support separate normalizations.

In [ ]:
from examples.hep_classifier import load_fixture, load_provenance

data = load_fixture()
provenance = load_provenance()

{
    "events": data.n_events,
    "signal (htautau)": int(data.is_signal.sum()),
    "background": int((~data.is_signal).sum()),
    "tes points": sorted(data.variants),
    "feature columns": len(data.feature_names),
    "tes-responsive features": len(provenance["tes_responsive_features"]),
    "licence": provenance["source_license"],
    "licence record DOI": provenance["license_record_doi"],
}

## The classifier-to-score bridge, live

Both classifiers are cross-fitted **out-of-fold with one fold id per event**, reused by both
`tes` copies of that event. Without that grouping the `tes` classifier memorizes an event from
its eighteen `tes`-inert columns and answers with the *opposite* label -- the design record
measured an out-of-fold AUC of 0.34 (below chance) with a plain stratified split, and 0.57 once
folds are grouped by event. Both classifiers train under the **Monte Carlo weights**, so the
ratios they estimate are those of the physical mixture; the signal/background classifier
additionally balances its two classes to a half each, because the raw weights make the signal
class statistically invisible (weighted fraction ~0.001), and the physical rate enters through
`IntensityParameterization` coefficients instead.

In [ ]:
from examples.hep_classifier.scores import (
    assemble_score_sample,
    event_folds,
    fit_signal_background_oof,
    fit_tes_oof,
)

fold_ids = event_folds(data.is_signal, n_folds=3, seed=2026)
sigbg = fit_signal_background_oof(data, fold_ids=fold_ids, max_iter=60, seed=2126)
tes = fit_tes_oof(data, delta=0.05, fold_ids=fold_ids, max_iter=60, seed=2526)
sample = assemble_score_sample(data, sigbg, tes)

{
    "score shape": tuple(sample.scores.shape),
    "schema": sample.schema.parameters,
    "provenance kind": sample.provenance.kind,
    "weighted signal AUC (out of fold)": round(sigbg.weighted_auc, 4),
    "weighted signal fraction": round(sigbg.signal_fraction, 5),
    "tes minus/plus AUC (out of fold)": round(tes.minus_plus_auc, 4),
}

## Two criteria, two partitions of the same sample

One call each; the only difference is the criterion object. Both labelings are scored twice, on
the whole three-parameter Fisher matrix and on the profiled information of `mu_htautau` alone --
the disagreement between the two columns is the entire point of a profiled criterion.

In [ ]:
import numpy as np

import scorequant as sq
from examples.hep_classifier.experiment import score_labeling

config = sq.DExchangeConfig(seed=11)
n_bins = 4

plain = sq.optimize_partition(sample, n_bins=n_bins, criterion=sq.DOptimality(), config=config)
profiled = sq.optimize_partition(
    sample, n_bins=n_bins, criterion=sq.ProfiledDOptimality(("mu_htautau",)), config=config
)

header = f"{'labeling':<16}{'full D':>10}{'profiled D_s':>15}"
print(header)
print("-" * len(header))
for name, result in (("plain D", plain), ("profiled D_s", profiled)):
    scored = score_labeling(sample.scores, np.asarray(result.labels), sample.weights, n_bins=n_bins)
    print(f"{name:<16}{scored.full_retention:>10.4f}{scored.profiled_retention:>15.4f}")

## The full committed study

`run_study` runs everything the doc page reports: the in-sample partitions on all events
against the classifier-output baselines and the certified `efficient_score_bound` ceiling, the
bin-budget sweep, the three-point `delta` convergence study, the cross-evaluation of every
reusable rule in both directions of a half/half event split, and the downstream Asimov
uncertainty on the signal strength from each rule's yield templates.

In [ ]:
from examples._env import is_fast_mode
from examples.hep_classifier.experiment import run_study

study = run_study(
    n_folds=3 if is_fast_mode() else 5,
    max_iter=60 if is_fast_mode() else 300,
    soft_steps=80 if is_fast_mode() else 400,
    budgets=(3, 6) if is_fast_mode() else (3, 4, 6, 8),
    bootstrap_replicates=10 if is_fast_mode() else 200,
)

in_sample = study.metrics["in_sample"]
held_out = study.metrics["cross_evaluation"]["mean"]
header = f"{'rule':<36}{'in-sample full':>16}{'in-sample profiled':>20}{'held-out profiled':>19}"
print(header)
print("-" * len(header))
for key, row in in_sample["by_key"].items():
    held = held_out.get(key, {}).get("evaluation_profiled_retention")
    full = "n/a" if row["full_retention"] is None else f"{row['full_retention']:.4f}"
    profiled = "n/a" if row["profiled_retention"] is None else f"{row['profiled_retention']:.4f}"
    held_text = "n/a" if held is None else f"{held:.4f}"
    print(f"{row['label']:<36}{full:>16}{profiled:>20}{held_text:>19}")

## The committed figures

In [ ]:
from examples.hep_classifier.figures import make_budget_figure, make_cells_figure

cells = make_cells_figure(study)
budget = make_budget_figure(study)
cells

## Interpretation

`mu_htautau` is a rate direction with a closed-form score once the classifier ratio is known;
`tes` has no closed form and is estimated by the library's central-difference door,
`CentralLogRatioScore`. Read the printed tables above rather than any number quoted in prose
here: this notebook runs in fast mode during CI and its exact figures move a little between
budgets and seeds, but the qualitative story is stable across runs -- the profiled criterion
trades away full-matrix retention for information about `mu_htautau` specifically, the
reusable profiled rule keeps most of that information on events it never saw, and a binning of
the classifier output that never looks at the `tes` column leaves a large, measured amount of
it on the table. The downstream entry of `study.metrics` states the same thing in the reported
quantity: with `tes` fixed every six-bin rule measures the signal strength about equally well,
and with `tes` floating only the profiled rule keeps its uncertainty. The doc page's evidence
JSON pins the exact full-scale numbers this notebook approximates.